# Lab 2.5 &mdash; Challenge &mdash; The Architecture Bake-Off

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Put all four Module 2 architectures behind one interface
- Write the acceptance bar before you look at a single result
- Run one eval set through all four and pick a winner on evidence
- Find the case that no architecture gets right, and say why

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The take-home artifact.** A harness you can point at your own task, and a habit:
> choose the reasoning architecture with a number, not a preference.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 2 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the two tools, carried through Module 2
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1003'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    """
    rec = LEDGER.get(ref)
    return json.dumps({"ref": ref, **rec}) if rec else f"no payment found with reference {ref!r}"


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {t.name: t for t in (lookup_payment, policy_for)}
print("tools:", list(TOOLS))

## Concept

You have built four ways to answer the same question:

| Arm | What it is | What it costs |
|---|---|---|
| **direct** | one chain, no reasoning asked for | one call |
| **cot** | one chain, working shown | one call, longer |
| **react** | `create_agent` with tools | several calls |
| **reflect** | draft, critique, revise | two to six calls |

The right answer is task-dependent and it is often **direct**. This lab is the harness that tells
you which, and the discipline of writing the bar down first so the harness can overrule you.

## Section 1 &mdash; One interface, four arms

Every arm is a function `(ref) -> str`. That is the whole contract, and it is what makes them
comparable.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent

SYSTEM = "You are a payments operations analyst."
QUESTION = "What must happen next with {ref}? Answer with the single next action."

def _context(ref: str) -> str:
    rec = LEDGER[ref]
    return (f"PAYMENT: {json.dumps({'ref': ref, **rec})}\n"
            f"POLICY: {POLICY.get(rec['reason_code'], 'no policy applies')}")

def arm_direct(ref: str) -> str:
    chain = ChatPromptTemplate.from_messages(
        [("system", SYSTEM), ("human", "{ctx}\n\n" + QUESTION.format(ref=ref))]
    ) | get_llm() | StrOutputParser()
    return chain.invoke({"ctx": _context(ref)})

def arm_cot(ref: str) -> str:
    chain = ChatPromptTemplate.from_messages(
        [("system", SYSTEM + " Work through the case step by step, then give the action."),
         ("human", "{ctx}\n\n" + QUESTION.format(ref=ref))]
    ) | get_llm() | StrOutputParser()
    return chain.invoke({"ctx": _context(ref)})

def arm_react(ref: str) -> str:
    agent = create_agent(model=get_llm(), tools=list(TOOLS.values()),
                         system_prompt=SYSTEM + " Use the tools to find the reason code and its "
                                                "policy before answering.")
    out = agent.invoke({"messages": [("human", QUESTION.format(ref=ref))]})
    return out["messages"][-1].content

def arm_reflect(ref: str) -> str:
    draft = arm_direct(ref)
    critique = ask(f"{_context(ref)}\n\nPROPOSED: {draft}",
                   system="List material faults against the policy, or reply exactly: NO ISSUES")
    if BLANK:                         # TODO: when is the draft already good enough to return?
        return draft
    return ask(f"{_context(ref)}\n\nPROPOSED: {draft}\nFAULTS: {critique}\n\n"
               "Rewrite the action in one sentence, fixing only what was faulted.")

ARMS = {"direct": arm_direct, "cot": arm_cot, "react": arm_react, "reflect": arm_reflect}

In [ ]:
# --- Self-check: Section 1   (structure only -- no model call)
check("all four arms are registered",
      lambda: set(ARMS) == {"direct", "cot", "react", "reflect"})
check("every arm is callable",
      lambda: all(callable(f) for f in ARMS.values()),
      "the whole comparison rests on the four having the same interface")
check("every arm takes exactly one argument",
      lambda: all(f.__code__.co_argcount == 1 for f in ARMS.values()))
check("the prompt-only arms share one context builder",
      lambda: all("_context" in ARMS[n].__code__.co_names for n in ("direct", "cot")),
      "if the arms build their input differently you are comparing prompts, not architectures")
check("the react arm gets its context from the tools instead",
      lambda: "create_agent" in ARMS["react"].__code__.co_names,
      "that IS the architectural difference -- it fetches rather than being handed the answer")

## Section 2 &mdash; The bar, written first

Fill this in before you run anything. If you write it afterwards you will write down whatever
you got.

In [ ]:
BAR = {
    "min_pass_rate":   0.80,          # a candidate must answer four of five
    "must_never_fail": ["PMT-1005"],  # the sanctions hold: getting this wrong is disqualifying
    "max_seconds_case": 20.0,
}

CASES = [
    {"ref": "PMT-1003", "must_contain": ["treasury"]},
    {"ref": "PMT-1005", "must_contain": ["compliance"]},
    {"ref": "PMT-1002", "must_contain": ["retry"]},
    {"ref": "PMT-1004", "must_contain": ["originator", "r04"]},
    {"ref": "PMT-1001", "must_contain": ["settled", "no action", "none"]},
]

def passes(case: dict, answer: str) -> bool:
    low = (answer or "").lower()
    return any(term in low for term in case["must_contain"])


def accepts(result: dict) -> tuple[bool, str]:
    """result: {"rate", "failed_refs", "seconds_case"}. Return (ok, first failing reason)."""
    if result["rate"] < BAR["min_pass_rate"]:
        return False, f"pass rate {result['rate']:.0%} below {BAR['min_pass_rate']:.0%}"
    banned = BLANK                    # TODO: did it fail a case it is not allowed to fail?
    if banned:
        return False, f"failed a disqualifying case: {sorted(banned)}"
    if result["seconds_case"] > BAR["max_seconds_case"]:
        return False, f"{result['seconds_case']:.1f}s/case over {BAR['max_seconds_case']}"
    return True, "accepted"

In [ ]:
# --- Self-check: Section 2
_good     = {"rate": 1.0, "failed_refs": [],            "seconds_case": 3.0}
_low      = {"rate": 0.6, "failed_refs": ["PMT-1002"],  "seconds_case": 3.0}
_unsafe   = {"rate": 0.8, "failed_refs": ["PMT-1005"],  "seconds_case": 3.0}
_slow     = {"rate": 1.0, "failed_refs": [],            "seconds_case": 99.0}

check("a clean result is accepted",   lambda: accepts(_good)[0] is True)
check("a low pass rate is rejected",  lambda: accepts(_low)[0] is False)
check("failing the sanctions case is disqualifying even at 80%",
      lambda: accepts(_unsafe)[0] is False,
      "some cases are not worth 20% -- they are worth the whole decision")
check("the rejection names the case",
      lambda: "PMT-1005" in accepts(_unsafe)[1])
check("a slow arm is rejected",       lambda: accepts(_slow)[0] is False)
check("the scorer accepts a right answer",
      lambda: passes(CASES[1], "Hold; Compliance decides.") is True)
check("the scorer rejects a wrong one",
      lambda: passes(CASES[1], "Release once Treasury approves.") is False)

## Section 3 &mdash; The bake-off

In [ ]:
def run_arm(name: str) -> dict:
    """Run every case through one arm and score it."""
    fn = ARMS[name]
    t0, answers, results, failed = time.time(), [], [], []
    for case in CASES:
        try:
            answer = fn(case["ref"])
        except Exception as exc:
            answer = f"<error: {type(exc).__name__}: {exc}>"
        ok = passes(case, answer)
        answers.append(answer); results.append(ok)
        if not ok:
            failed.append(case["ref"])
    seconds = time.time() - t0
    return {"name": name, "answers": answers, "results": results, "failed_refs": failed,
            "rate": sum(results) / len(CASES), "seconds_case": seconds / len(CASES)}

In [ ]:
# --- Self-check: Section 3   (shape of the result, on a stub arm -- no model call)
ARMS["_stub"] = lambda ref: {"PMT-1005": "Hold; Compliance decides."}.get(ref, "do something")
_stub = run_arm("_stub")
del ARMS["_stub"]

check("one answer per case",       lambda: len(_stub["answers"]) == len(CASES))
check("the rate is a fraction",    lambda: 0.0 <= _stub["rate"] <= 1.0)
check("failed refs are recorded",  lambda: "PMT-1003" in _stub["failed_refs"])
check("a passing case is not listed as failed",
      lambda: "PMT-1005" not in _stub["failed_refs"])
def _raises_is_scored():
    def boom(ref): raise RuntimeError("nope")
    ARMS["_boom"] = boom
    try:
        return run_arm("_boom")["rate"] == 0.0
    finally:
        del ARMS["_boom"]

check("an arm that raises is scored, not crashed",
      lambda: _raises_is_scored(),
      "one broken arm must not take the whole bake-off down with it")

## Run it for real

Four arms, five cases. `react` and `reflect` make several calls per case, so allow a minute or two.

In [ ]:
if llm_ready():
    def _bakeoff():
        results = {name: run_arm(name) for name in ("direct", "cot", "react", "reflect")}

        print("  arm       " + "  ".join(f"{c['ref']:9}" for c in CASES) + "   rate    s/case  verdict")
        print("  " + "-" * 96)
        for name, r in results.items():
            cells = "  ".join(("pass     " if ok else "FAIL     ") for ok in r["results"])
            ok, why = accepts(r)
            print(f"  {name:9} {cells}  {r['rate']:5.0%}  {r['seconds_case']:6.1f}  "
                  f"{'ACCEPT' if ok else 'reject'}")
            if not ok:
                print(f"             -> {why}")

        accepted = [(n, r) for n, r in results.items() if accepts(r)[0]]
        if accepted:
            # Cheapest that cleared the bar -- but a tie on cost is broken by pass rate.
            # Paying nothing extra for a better answer is not a reason to refuse it.
            winner = min(accepted, key=lambda p: (round(p[1]["seconds_case"], 1), -p[1]["rate"]))
            print(f"\nwinner: {winner[0]} -- the CHEAPEST arm that cleared the bar, "
                  f"not the best-scoring one (cost ties broken on pass rate)")
        else:
            print("\nno arm cleared the bar. That is a result: the task needs better tools or "
                  "a better prompt, not a fancier architecture.")

        hard = [c["ref"] for i, c in enumerate(CASES)
                if not any(r["results"][i] for r in results.values())]
        print(f"cases no arm answered: {hard or 'none'}")
        return results
    BAKEOFF = guard(_bakeoff)

### Read it

**The winner is the cheapest arm that cleared the bar**, not the highest-scoring one. That line in
the code is the whole lab. Once an arm meets the requirement, further quality is something you are
paying for and not using &mdash; and `direct` clearing the bar is the most common outcome on tasks
where the context already contains the answer.

Note the tie-break, though: when two arms cost the *same*, the higher pass rate wins. "Prefer the
cheaper design" is an argument about what you are willing to pay for, not a reason to accept a
worse answer that costs nothing extra. Those are different claims and it is worth being able to
tell them apart in a design review.

Then look at the last line. A case that **no** arm answers is telling you something no
architecture can fix: the information is missing, the scorer is wrong, or the question is
ambiguous. Reaching for a bigger architecture at that point is the mistake this module exists to
prevent.

**What you take from Module 2:** chains you compose rather than hand-roll; a parser you are now
entitled never to write again; plans that are validated before they run and failures that are
diagnosed before they are retried; a judge that returns numbers; and a harness that picks the
architecture for you. Module 3 gives all of this somewhere durable to live.

In [ ]:
score()

## Your turn

1. Add a `react_reflect` arm &mdash; the agent's answer, then one critique round. Does it clear the
   bar, and does it beat `react` by enough to justify doubling the calls?
2. Move `PMT-1001` (already settled) to the front of `CASES` and re-run. If any arm's score
   changes, you have found order dependence, which means your harness is measuring the wrong
   thing. Fix it.
3. Replace the substring scorer with a `with_structured_output` judge from Lab 2.4 and re-run the
   bake-off. Which arms change rank? A scorer swap that reorders the results tells you the
   original ranking was never about the architectures.